In [ ]:
!pip install scikeras

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, LearningCurveDisplay
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Molecules_Toxicity_Classification.csv to Molecules_Toxicity_Classification.csv


In [ ]:
import io
import pandas as pd

file_path = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_path]))
df.head()

,Unnamed: 0,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,...,WTPT-4,WTPT-5,ETA_EtaP_L,ETA_EtaP_F,ETA_EtaP_B,nT5Ring,SHdNH,ETA_dEpsilon_C,MDEO-22,Class
0,0,0.0908,0,0.0075,0.0173,0,0.0,-0.0436,0.0409,0.0000,...,0.0000,0.0000,0.1780,1.5488,0.0088,0,0.0,-0.0868,0.00,NonToxic
1,1,0.0213,0,0.1144,-0.0410,0,0.0,0.1231,-0.0316,0.0000,...,8.8660,19.3525,0.1739,1.3718,0.0048,2,0.0,-0.0810,0.25,NonToxic
2,2,0.0018,0,-0.0156,-0.0765,2,0.0,-0.1138,-0.1791,0.0000,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0.0,-0.1004,0.00,NonToxic
3,3,-0.0251,0,-0.0064,-0.0894,3,0.0,-0.0747,-0.1151,0.0000,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0.0,-0.1010,0.00,NonToxic
4,4,-0.0028,2,-0.0164,-0.0912,2,0.0,-0.0356,-0.0159,6.0139,...,15.6022,6.2113,0.1755,1.4774,0.0133,0,0.0,-0.0980,0.00,NonToxic


In [ ]:
df.shape

(159, 1205)

In [ ]:
df.dtypes

,0
Unnamed: 0,int64
MATS3v,float64
nHBint10,int64
MATS3s,float64
MATS3p,float64
...,...
nT5Ring,int64
SHdNH,float64
ETA_dEpsilon_C,float64
MDEO-22,float64


In [ ]:
df.drop(columns=[df.columns[0]], inplace=True)

In [ ]:
label_encoder = LabelEncoder()
df['Class'] = label_encoder.fit_transform(df['Class'])  # 0: NonToxic, 1: Toxic

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam, RMSprop

In [ ]:
from tensorflow.keras.optimizers import Adam, RMSprop

def create_model(learning_rate=0.001, optimizer='adam'):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    if optimizer == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("Invalid optimizer")

    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['AUC'])
    return model

In [ ]:
from scikeras.wrappers import KerasClassifier

nn_clf = KerasClassifier(model=create_model, verbose=0)

In [ ]:
NN_params = {
    "classifier__model__learning_rate": [0.001, 0.0001],
    "classifier__model__optimizer": ['adam', 'rmsprop']
}

In [ ]:
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
nn_clf = KerasClassifier(model=create_model, verbose=0)

NN_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('classifier', nn_clf)
])


In [ ]:
cv = 5
NN_grid_search = GridSearchCV(NN_pipeline, NN_params, scoring="roc_auc", cv=cv)
NN_grid_search.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier',
                                        KerasClassifier(model=<function create_model at 0x7a2cc0e63420>, verbose=0))]),
             param_grid={'classifier__model__learning_rate': [0.001, 0.0001],
                         'classifier__model__optimizer': ['adam', 'rmsprop']},
             scoring='roc_auc')

In [ ]:
nn_best = NN_grid_search.best_estimator_
print(nn_best)


Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 KerasClassifier(model=<function create_model at 0x7a2cc0e63420>, model__learning_rate=0.001, model__optimizer='adam', verbose=0))])


In [ ]:
print(NN_grid_search.best_params_)

NameError: name 'NN_grid_search' is not defined

In [ ]:
best_lr = NN_grid_search.best_params_["classifier__model__learning_rate"]
best_optimizer = NN_grid_search.best_params_["classifier__model__optimizer"]
print(f"Best Learning Rate: {best_lr}")
print(f"Best Optimizer: {best_optimizer}")


Best Learning Rate: 0.001
Best Optimizer: adam


In [ ]:
NN_grid_search.fit(X_train, y_train)
nn_best = NN_grid_search.best_estimator_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^

In [ ]:
print(NN_grid_search.best_params_)
print(NN_grid_search.best_score_)

{'classifier__model__learning_rate': 0.001, 'classifier__model__optimizer': 'adam'}
nan


In [ ]:
import numpy as np

print("Unique values in y_train:", np.unique(y_train))
print("Any NaNs in y_train:", np.isnan(y_train).any())

Unique values in y_train: [0 1]
Any NaNs in y_train: False


In [ ]:
import pandas as pd
pd.DataFrame(NN_grid_search.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__model__learning_rate,param_classifier__model__optimizer,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,1.779067,0.177653,0.001260,0.000206,0.0010,adam,"{'classifier__model__learning_rate': 0.001, 'c...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,1.368082,0.222979,0.001166,0.000058,0.0010,rmsprop,"{'classifier__model__learning_rate': 0.001, 'c...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2.151893,0.677542,0.001601,0.000403,0.0001,adam,"{'classifier__model__learning_rate': 0.0001, '...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,3.112689,1.267611,0.003707,0.001469,0.0001,rmsprop,"{'classifier__model__learning_rate': 0.0001, '...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [ ]:
NN_grid_search = GridSearchCV(NN_pipeline, NN_params, scoring="roc_auc", cv=3)
NN_grid_search.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier',
                                        KerasClassifier(model=<function create_model at 0x7a2cc0e63420>, verbose=0))]),
             param_grid={'classifier__model__learning_rate': [0.001, 0.0001],
                         'classifier__model__optimizer': ['adam', 'rmsprop']},
             scoring='roc_auc')

In [ ]:
if hasattr(nn_best, "predict_proba"):
    print("predict_proba is available")
else:
    print("predict_proba is NOT available")

predict_proba is available


In [ ]:
nn_model = nn_best.named_steps["classifier"]  # Extract the final estimator
nn_y_pred_proba = nn_model.predict_proba(X_test)  # Get probability estimates
nn_y_pred = (nn_y_pred_proba[:, 1] >= 0.6).astype(int)  # Apply threshold

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, nn_y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7188


In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, nn_y_pred))

[[23  1]
 [ 8  0]]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, nn_y_pred))

              precision    recall  f1-score   support

           0       0.74      0.96      0.84        24
           1       0.00      0.00      0.00         8

    accuracy                           0.72        32
   macro avg       0.37      0.48      0.42        32
weighted avg       0.56      0.72      0.63        32



In [ ]:
from sklearn.metrics import roc_auc_score

auc_score = roc_auc_score(y_test, nn_y_pred_proba[:, 1])
print(f"AUC-ROC: {auc_score:.4f}")

AUC-ROC: 0.4844



# Trial 1

In [ ]:
!pip install scikeras

from imblearn.pipeline import Pipeline as ImbPipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, LearningCurveDisplay
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Molecules_Toxicity_Classification.csv to Molecules_Toxicity_Classification (1).csv


In [ ]:
import io
import pandas as pd

file_path = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_path]))
df.head()

,Unnamed: 0,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,...,WTPT-4,WTPT-5,ETA_EtaP_L,ETA_EtaP_F,ETA_EtaP_B,nT5Ring,SHdNH,ETA_dEpsilon_C,MDEO-22,Class
0,0,0.0908,0,0.0075,0.0173,0,0.0,-0.0436,0.0409,0.0000,...,0.0000,0.0000,0.1780,1.5488,0.0088,0,0.0,-0.0868,0.00,NonToxic
1,1,0.0213,0,0.1144,-0.0410,0,0.0,0.1231,-0.0316,0.0000,...,8.8660,19.3525,0.1739,1.3718,0.0048,2,0.0,-0.0810,0.25,NonToxic
2,2,0.0018,0,-0.0156,-0.0765,2,0.0,-0.1138,-0.1791,0.0000,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0.0,-0.1004,0.00,NonToxic
3,3,-0.0251,0,-0.0064,-0.0894,3,0.0,-0.0747,-0.1151,0.0000,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0.0,-0.1010,0.00,NonToxic
4,4,-0.0028,2,-0.0164,-0.0912,2,0.0,-0.0356,-0.0159,6.0139,...,15.6022,6.2113,0.1755,1.4774,0.0133,0,0.0,-0.0980,0.00,NonToxic


In [ ]:
df.shape

(159, 1205)

In [ ]:
df.dtypes

,0
Unnamed: 0,int64
MATS3v,float64
nHBint10,int64
MATS3s,float64
MATS3p,float64
...,...
nT5Ring,int64
SHdNH,float64
ETA_dEpsilon_C,float64
MDEO-22,float64


In [ ]:
label_encoder = LabelEncoder()
df['Class'] = label_encoder.fit_transform(df['Class'])  # 0: NonToxic, 1: Toxic

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout

def create_model(learning_rate=0.001, optimizer='adam'):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    if optimizer == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("Invalid optimizer")

    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['AUC'])
    return model


In [ ]:
model = create_model(learning_rate=0.001, optimizer='adam')
model.fit(X_train, y_train, class_weight=class_weights_dict, epochs=20, batch_size=32)

Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - AUC: 0.5053 - loss: 0.8868
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8138 - loss: 0.5078
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9044 - loss: 0.4038
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9602 - loss: 0.3301
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.9378 - loss: 0.3429
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.9873 - loss: 0.2528
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.9865 - loss: 0.2362
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - AUC: 0.9770 - loss: 0.2477
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.9868 - loss: 0.2008
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9873 - loss: 0.1869
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9941 - loss: 0.1588
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9974 - loss: 0.1475 
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9988 - loss:


# Trial 2

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Resampled class distribution:", np.bincount(y_train_resampled))


Resampled class distribution: [84 84]


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}
print("Class Weights:", class_weights_dict)

Class Weights: {0: np.float64(0.7559523809523809), 1: np.float64(1.4767441860465116)}


In [ ]:
model = create_model(learning_rate=0.001, optimizer='adam')
model.fit(X_train_resampled, y_train_resampled, class_weight=class_weights_dict, epochs=20, batch_size=32)

Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - AUC: 0.5549 - loss: 1.1426
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - AUC: 0.8913 - loss: 0.4677
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.9280 - loss: 0.3867
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - AUC: 0.9527 - loss: 0.3416 
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - AUC: 0.9806 - loss: 0.2636
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - AUC: 0.9843 - loss: 0.2451
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - AUC: 0.9911 - loss: 0.1988
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.9830 - loss: 0.2085
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - AUC: 0.9948 - loss: 0.1371 
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - AUC: 0.9895 - loss: 0.1722 
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - AUC: 0.9991 - loss: 0.1140
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - AUC: 0.9959 - loss: 0.1213
Epoch 13/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - AUC: 0.9986 - loss: 

In [ ]:
def create_model(learning_rate=0.001, optimizer='adam'):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    if optimizer == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("Invalid optimizer")

    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['AUC'])
    return model

In [ ]:
NN_params = {
    "classifier__model__learning_rate": [0.01, 0.001, 0.0001],
    "classifier__model__optimizer": ['adam', 'rmsprop'],
    "classifier__batch_size": [16, 32, 64],
    "classifier__epochs": [10, 20, 30]
}

In [ ]:
NN_grid_search = GridSearchCV(NN_pipeline, NN_params, scoring="f1", cv=5)

In [ ]:
nn_y_pred_proba = model.predict(X_test)
nn_y_pred = (nn_y_pred_proba >= 0.6).astype(int)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step


In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, nn_y_pred)
print(f" Accuracy: {accuracy:.4f}")

 Accuracy: 0.5938


In [ ]:
from sklearn.metrics import confusion_matrix
print(" Confusion Matrix:")
print(confusion_matrix(y_test, nn_y_pred))

 Confusion Matrix:
[[18  6]
 [ 7  1]]


In [ ]:
from sklearn.metrics import classification_report
print(" Classification Report:")
print(classification_report(y_test, nn_y_pred))

 Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.75      0.73        24
           1       0.14      0.12      0.13         8

    accuracy                           0.59        32
   macro avg       0.43      0.44      0.43        32
weighted avg       0.58      0.59      0.58        32



In [ ]:
from sklearn.metrics import roc_auc_score
auc_score = roc_auc_score(y_test, nn_y_pred_proba)
print(f" AUC-ROC: {auc_score:.4f}")

 AUC-ROC: 0.5885


# Trial 3

In [ ]:
nn_y_pred = (nn_y_pred_proba >= 0.5).astype(int)

In [ ]:
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

def create_model(learning_rate=0.001, optimizer='adam'):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    opt = Adam(learning_rate=learning_rate) if optimizer == 'adam' else RMSprop(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['AUC'])
    return model


In [ ]:
class_weights_dict = {0: 1, 1: 3}

In [ ]:
model = create_model(learning_rate=0.0005, optimizer='adam')
history = model.fit(X_train, y_train, epochs=50, batch_size=16, class_weight=class_weights_dict, validation_data=(X_test, y_test))

Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - AUC: 0.5960 - loss: 1.3641 - val_AUC: 0.3750 - val_loss: 0.9023
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - AUC: 0.7399 - loss: 1.0971 - val_AUC: 0.3385 - val_loss: 0.9564
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - AUC: 0.8318 - loss: 0.8263 - val_AUC: 0.3672 - val_loss: 0.9793
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - AUC: 0.8742 - loss: 0.6987 - val_AUC: 0.3698 - val_loss: 0.9647
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - AUC: 0.8336 - loss: 0.7863 - val_AUC: 0.3802 - val_loss: 1.0051
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.8284 - loss: 0.8974 - val_AUC: 0.3958 - val_loss: 1.0081
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.9612 - loss: 0.5045 - val_AUC: 0.4219 - val_loss: 0.9897
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - AUC: 0.8572 - loss: 0.7648 - val_AUC: 0.4349 - val_loss: 0.9609
Epoch 9/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - AUC: 0.9686 - loss: 0.4564 - val_AUC: 0.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

nn_y_pred_proba = model.predict(X_test)  # Get probability predictions
nn_y_pred = (nn_y_pred_proba >= 0.5).astype(int)

# Print performance metrics
print("Confusion Matrix:\n", confusion_matrix(y_test, nn_y_pred))
print("\nClassification Report:\n", classification_report(y_test, nn_y_pred))
print("\nAUC-ROC:", roc_auc_score(y_test, nn_y_pred_proba))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Confusion Matrix:
 [[17  7]
 [ 4  4]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.71      0.76        24
           1       0.36      0.50      0.42         8

    accuracy                           0.66        32
   macro avg       0.59      0.60      0.59        32
weighted avg       0.70      0.66      0.67        32


AUC-ROC: 0.6458333333333333


Since recall for class 1 improved, but precision is still low, we can try lowering the threshold to 0.35 or 0.3 to capture more true positives.

In [ ]:
nn_y_pred = (nn_y_pred_proba >= 0.35).astype(int)

# Re-evaluate the model
print("Confusion Matrix:\n", confusion_matrix(y_test, nn_y_pred))
print("\nClassification Report:\n", classification_report(y_test, nn_y_pred))
print("\nAUC-ROC:", roc_auc_score(y_test, nn_y_pred_proba))

Confusion Matrix:
 [[17  7]
 [ 2  6]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.71      0.79        24
           1       0.46      0.75      0.57         8

    accuracy                           0.72        32
   macro avg       0.68      0.73      0.68        32
weighted avg       0.79      0.72      0.74        32


AUC-ROC: 0.6458333333333333


In [ ]:
class_weights_dict = {0: 1, 1: 4}

In [ ]:
history = model.fit(X_train, y_train, epochs=100, batch_size=32, class_weight=class_weights_dict, validation_data=(X_test, y_test))

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 230ms/step - AUC: 0.9822 - loss: 0.2510 - val_AUC: 0.6536 - val_loss: 1.0137
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - AUC: 0.9965 - loss: 0.1551 - val_AUC: 0.6458 - val_loss: 1.0173
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - AUC: 0.9930 - loss: 0.2550 - val_AUC: 0.6380 - val_loss: 1.1065
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - AUC: 0.9998 - loss: 0.1197 - val_AUC: 0.6380 - val_loss: 1.1573
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - AUC: 1.0000 - loss: 0.0958 - val_AUC: 0.6380 - val_loss: 1.1867
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - AUC: 0.9950 - loss: 0.1215 - val_AUC: 0.6380 - val_loss: 1.2126
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - AUC: 0.9996 - loss: 0.1253 - val_AUC: 0.6198 - val_loss: 1.2280
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - AUC: 0.9989 - loss: 0.1407 - val_AUC: 0.6146 - val_loss: 1.2123
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - AUC: 0.9998 - loss:

In [ ]:
nn_y_pred_proba = nn_best.named_steps["classifier"].predict_proba(X_test)
nn_y_pred = (nn_y_pred_proba[:, 1] >= 0.5).astype(int)  # Change threshold if needed

ValueError: X has shape (1204,), but this KerasClassifier is expecting X of shape (1203,)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (127, 1204)
X_test shape: (32, 1204)


In [ ]:
print("Expected input shape:", nn_best.named_steps["classifier"].model_.input_shape)

Expected input shape: (None, 1203)


In [ ]:
X_train = X_train[:, :1203]  # Keep only the first 1203 features
X_test = X_test[:, :1203]

In [ ]:
import numpy as np

X_train = np.array(X_train)
X_test = np.array(X_test)

X_train = X_train.reshape(-1, 1203)  # Ensure shape is (num_samples, num_features)
X_test = X_test.reshape(-1, 1203)

In [ ]:
nn_y_pred_proba = nn_best.named_steps["classifier"].predict_proba(X_test)
nn_y_pred = (nn_y_pred_proba[:, 1] >= 0.5).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, nn_y_pred)
print(f"Accuracy: {accuracy:.4f}")

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, nn_y_pred)
print("Confusion Matrix:\n", cm)

from sklearn.metrics import classification_report

print("Classification Report:\n", classification_report(y_test, nn_y_pred))

from sklearn.metrics import roc_auc_score

auc_score = roc_auc_score(y_test, nn_y_pred_proba[:, 1])
print(f"AUC-ROC: {auc_score:.4f}")

Accuracy: 0.5625
Confusion Matrix:
 [[18  6]
 [ 8  0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.75      0.72        24
           1       0.00      0.00      0.00         8

    accuracy                           0.56        32
   macro avg       0.35      0.38      0.36        32
weighted avg       0.52      0.56      0.54        32

AUC-ROC: 0.4688
